## Ingest Results Data (JSON to Delta Lake)

This notebook reads the **results** JSON files (containing Formula 1 race results - who finished where, how many laps, points scored, etc.) from the landing volume and loads them into a Bronze Delta table.

**What's different here?** The results data comes as **multiple JSON files in a folder** (not a single file). Spark handles this automatically - just point `.load()` to the folder and it reads all files inside.

**Steps:**
1. **Define schema** and **read** the JSON files
2. **Enrich** with metadata columns (ingestion timestamp + source file)
3. **Write** to the Bronze Delta table `formula1.bronze.results`
4. **Verify** the data was written correctly

#### Loading Configuration
We import shared variables and helper functions from the `00-common` folder.

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

#### Step 4: Verify the Data
As a final check, we read back the table we just wrote to confirm everything landed correctly. If you see rows with all columns populated, the ingestion worked!

In [0]:
source_file = f'{loding_folder_path}/results' # to replace the load data in read api
table_name = f'{catalog_name}.{bronze_schema}.results' # to replace the save table in write api 

#### Verify
As a final check, we read back the table to confirm all results data landed correctly.

#### Schema
We define the schema as a **DDL string** listing all 14 columns:
- Race info: `season`, `round`, `raceName`, `date`, `url`
- Driver/team: `driverId`, `constructorId`
- Result details: `grid` (starting position), `position` (finish), `positionText`, `points`, `laps`, `number`, `status`

**Note:** `points` is a `DOUBLE` (decimal) because half-points can be awarded. The rest are integers or strings.

In [0]:
from pyspark.sql.types import *
results_schema = ' date DATE,raceName string, round int, season int, url string, constructorId string, driverId string, grid int, laps int, number int, points double, position int, positionText string, status string'

#### Read API
We use Spark's DataFrame Reader to load all JSON files from the `results` folder.

**What's happening:**
- `format('json')` - tells Spark the files are JSON
- `.schema(results_schema)` - applies our DDL schema with 14 columns
- `.load(source_file)` - reads ALL JSON files from the `/results` folder

**Note:** Since `source_file` points to a folder (not a single file), Spark automatically reads every JSON file inside it and combines them into one DataFrame.

In [0]:
results_df = (
    spark.read
    .format('json')
    #.option('Headers',True)
    .schema(results_schema)
    .load(source_file)
)
display(results_df)


#### Metadata
We call `add_ingestion_metadata()` to add two tracking columns:
- **`ingestion_timestamp`** - when this data was loaded
- **`source_file`** - which file each row came from (especially useful here since there are multiple files)

In [0]:
results_final_df = add_ingestion_metadata(results_df)


#### Writing Delta Table
We save the enriched DataFrame as `formula1.bronze.results`:
- `mode('overwrite')` - replaces the entire table each run (clean reload)
- `format('delta')` - Delta format (versioning, time travel, fast queries)
- `saveAsTable(table_name)` - registers in Unity Catalog for SQL access

In [0]:
(
    results_final_df
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .format('delta')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))
